# 01 - HAM10000 ResNet50 Training Baseline

Purpose: train or retrain a binary ResNet50 baseline on HAM10000 using patient-level splitting where metadata allows it, class-imbalance handling, ROC-AUC monitoring, and checkpoint export.

This notebook is separate from the existing RQ notebooks. It produces training metrics that can be summarized by `PAPER_RESULTS_TRAINING_ADDENDUM.ipynb`.

Before running this notebook, run `00_setup_and_sanity.ipynb` once so `metadata_with_paths.csv` exists and image paths are prepared.

In [4]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
RESEARCH_DIR = NOTEBOOK_DIR.parent
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

from research_paths import METADATA_PATH, MODEL_DIR

OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("expected metadata:", METADATA_PATH)
print("model dir:", MODEL_DIR)

expected metadata: C:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_Classification_backend\ml\data\processed\metadata_with_paths.csv
model dir: C:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_Classification_backend\ml\outputs\models


In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

metadata_candidates = [
    METADATA_PATH,
    RESEARCH_DIR / "data" / "processed" / "metadata_with_paths.csv",
    RESEARCH_DIR / "data" / "metadata_with_paths.csv",
    NOTEBOOK_DIR / "data" / "processed" / "metadata_with_paths.csv",
]
metadata_file = next((path for path in metadata_candidates if path.exists()), None)
if metadata_file is None:
    checked = "\n".join(f"- {path}" for path in metadata_candidates)
    raise FileNotFoundError(
        "HAM10000 metadata was not found. Run 00_setup_and_sanity.ipynb first to generate "
        "metadata_with_paths.csv, or place the file in one of these locations:\n" + checked
    )

print("using metadata:", metadata_file)
df = pd.read_csv(metadata_file)
path_candidates = ["image_path", "filepath", "file_path", "path"]
path_col = next((col for col in path_candidates if col in df.columns), None)
if path_col is None:
    raise ValueError(f"Expected one image path column from {path_candidates}. Found columns: {list(df.columns)}")

if "binary_label" in df.columns:
    df["target"] = df["binary_label"].astype(int)
elif "target" in df.columns:
    df["target"] = df["target"].astype(int)
elif "label" in df.columns:
    df["target"] = df["label"].astype(int)
elif "dx" in df.columns:
    df["target"] = df["dx"].isin(["mel", "bcc", "akiec"]).astype(int)
else:
    raise ValueError(f"Expected binary_label, target, label, or dx column. Found columns: {list(df.columns)}")

df[path_col] = df[path_col].astype(str)
before_path_filter = len(df)
df = df[df[path_col].map(lambda p: Path(p).exists())].reset_index(drop=True)
if df.empty:
    raise FileNotFoundError(
        f"Metadata loaded from {metadata_file}, but none of the {before_path_filter} image paths in "
        f"column '{path_col}' exist on disk. Re-run 00_setup_and_sanity.ipynb so it writes current absolute paths."
    )

group_col = "lesion_id" if "lesion_id" in df.columns else ("patient_id" if "patient_id" in df.columns else None)
if group_col:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(splitter.split(df, df["target"], groups=df[group_col]))
else:
    train_idx, val_idx = train_test_split(
        np.arange(len(df)), test_size=0.2, random_state=42, stratify=df["target"]
    )

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print("rows:", len(df), "train:", len(train_df), "val:", len(val_df))
print("positive rate train:", train_df.target.mean().round(4), "val:", val_df.target.mean().round(4))
if group_col:
    overlap = set(train_df[group_col]).intersection(set(val_df[group_col]))
    print("group column:", group_col, "overlap:", len(overlap))

using metadata: C:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_Classification_backend\ml\data\processed\metadata_with_paths.csv
rows: 10015 train: 7991 val: 2024
positive rate train: 0.1628 val: 0.1611
group column: lesion_id overlap: 0


In [6]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class LesionDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row[path_col]).convert("RGB")
        return self.transform(image), torch.tensor(row["target"], dtype=torch.float32)

train_ds = LesionDataset(train_df, train_tfms)
val_ds = LesionDataset(val_df, val_tfms)
class_counts = train_df["target"].value_counts().sort_index()
weights = train_df["target"].map(lambda y: 1.0 / class_counts.loc[y]).to_numpy()
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(DEVICE)
print("device:", DEVICE)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\saiyu/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:07<00:00, 14.3MB/s]


device: cuda


In [7]:
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
from tqdm.auto import tqdm

EPOCHS = 3  # increase to 15+ for full training
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4, epochs=EPOCHS, steps_per_epoch=len(train_loader)
)
criterion = nn.BCEWithLogitsLoss()

def evaluate(model, loader):
    model.eval()
    probs, labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x).squeeze(1)
            loss = criterion(logits, y)
            total_loss += loss.item() * len(y)
            probs.extend(torch.sigmoid(logits).cpu().numpy().tolist())
            labels.extend(y.cpu().numpy().tolist())
    pred = [int(p >= 0.5) for p in probs]
    auc = roc_auc_score(labels, probs) if len(set(labels)) > 1 else float("nan")
    return {
        "loss": total_loss / len(loader.dataset),
        "roc_auc": auc,
        "accuracy": accuracy_score(labels, pred),
        "balanced_accuracy": balanced_accuracy_score(labels, pred),
    }

history = []
best_auc = -1
best_path = MODEL_DIR / "ham10000_resnet50_binary_best.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x).squeeze(1)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_loss += loss.item() * len(y)
    metrics = evaluate(model, val_loader)
    metrics.update({"epoch": epoch, "train_loss": train_loss / len(train_loader.dataset)})
    history.append(metrics)
    print(metrics)
    if metrics["roc_auc"] > best_auc:
        best_auc = metrics["roc_auc"]
        torch.save({"model_state_dict": model.state_dict(), "metrics": metrics}, best_path)

history_df = pd.DataFrame(history)
history_df.to_csv(METRICS_DIR / "TRAINING_ham10000_resnet50_baseline.csv", index=False)
print("saved metrics:", METRICS_DIR / "TRAINING_ham10000_resnet50_baseline.csv")
print("saved best checkpoint:", best_path)

c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_XAI_research\skin-lesion-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
epoch 1/3: 100%|██████████| 250/250 [02:25<00:00,  1.71it/s]


{'loss': 0.4182027735022217, 'roc_auc': np.float64(0.8569392356218432), 'accuracy': 0.7959486166007905, 'balanced_accuracy': np.float64(0.7705691286031202), 'epoch': 1, 'train_loss': 0.4905242860138215}


epoch 2/3: 100%|██████████| 250/250 [02:22<00:00,  1.76it/s]


{'loss': 0.3554216137987823, 'roc_auc': np.float64(0.8950461387269035), 'accuracy': 0.8364624505928854, 'balanced_accuracy': np.float64(0.7872795132490769), 'epoch': 2, 'train_loss': 0.35629690531839375}


epoch 3/3: 100%|██████████| 250/250 [02:56<00:00,  1.42it/s]


{'loss': 0.3027976471268141, 'roc_auc': np.float64(0.913564857970763), 'accuracy': 0.8710474308300395, 'balanced_accuracy': np.float64(0.8202847810849285), 'epoch': 3, 'train_loss': 0.24556995645778945}
saved metrics: c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_XAI_research\notebooks\outputs\metrics\TRAINING_ham10000_resnet50_baseline.csv
saved best checkpoint: C:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_Classification_backend\ml\outputs\models\ham10000_resnet50_binary_best.pth
